# 311 — Agent Setup

Bootstrap for the 31x interactive investigation notebook series.

This notebook is `%run` at the top of every other 31x notebook.
It establishes:
- Environment variables and API clients
- Neo4j connection health check
- Combined MCP tool list (Neo4j MCP + FastMCP)
- `execute_tool()` dispatcher used by all agents
- Shared `GRAPH_SCHEMA_HINT` constant

**Prerequisites:** Neo4j AuraDB seeded with Layers 1 & 2 data (run notebooks 111–216 first).

In [1]:
import sys, os, json, logging
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = Path(os.getcwd()).parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / '.env')

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s',
)
logging.getLogger('httpx').setLevel(logging.WARNING)
logging.getLogger('openai').setLevel(logging.WARNING)
print('Environment loaded.')

Environment loaded.


## Step 1 — Neo4j connection + health check

In [2]:
from src.graph.connection import Neo4jConnection

conn = Neo4jConnection()
conn.connect()
print('Connected to Neo4j.')

# Health check — verify key node labels exist
label_counts = conn.run_query("""
    CALL apoc.meta.stats() YIELD labels
    RETURN labels
""")

# Fallback if APOC not available
if not label_counts:
    label_counts = conn.run_query("""
        MATCH (n)
        WITH labels(n) AS lbs
        UNWIND lbs AS label
        RETURN label, count(*) AS count
        ORDER BY label
    """)

print('\nNode counts in graph:')
for row in label_counts:
    print(f'  {row}')

# Verify key Layer 1 and Layer 2 nodes are present
for label in ['Borrower', 'LoanApplication', 'BankAccount', 'Transaction',
               'Regulation', 'Section', 'Requirement', 'Threshold', 'Chunk']:
    result = conn.run_query(f'MATCH (n:{label}) RETURN count(n) AS n')
    count = result[0]['n'] if result else 0
    status = '✓' if count > 0 else '✗ MISSING'
    print(f'  {status}  {label}: {count}')

2026-03-09 14:26:50,266 [INFO] src.graph.connection: Connected to Neo4j at neo4j+s://44f9022d.databases.neo4j.io


Connected to Neo4j.

Node counts in graph:
  {'labels': {'CommercialSecured': 2, 'Federal': 1, 'Address': 609, 'Residential': 582, 'Corporate': 31, 'Collateral': 466, 'ResidentialSecured': 464, 'Industry': 14, 'National': 5, 'PersonalTransaction': 610, 'Threshold': 157, 'CorporateCurrent': 6, 'Borrower': 628, 'BusinessTransaction': 175, 'SpecialAdminRegion': 1, 'BankAccount': 791, 'Chunk': 205, 'Section': 118, 'Jurisdiction': 7, 'Commercial': 27, 'Requirement': 194, 'Transaction': 173, 'LoanApplication': 466, 'Individual': 597, 'Regulation': 3, 'Officer': 19}}
  ✓  Borrower: 628
  ✓  LoanApplication: 466
  ✓  BankAccount: 791
  ✓  Transaction: 173
  ✓  Regulation: 3
  ✓  Section: 118
  ✓  Requirement: 194
  ✓  Threshold: 157
  ✓  Chunk: 205


## Step 2 — API clients

In [ ]:
import anthropic
from openai import OpenAI
from src.agent.config import MODEL

claude_client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
oai_client    = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

print(f'Anthropic client ready. Model: {MODEL}')
print(f'OpenAI client ready.')

## Step 3 — Build combined MCP tool list

In [ ]:
from src.mcp.tool_defs import NEO4J_MCP_TOOLS, FASTMCP_TOOL_DEFS, TOOLS

print(f'Tool registry: {len(NEO4J_MCP_TOOLS)} Neo4j MCP + {len(FASTMCP_TOOL_DEFS)} FastMCP = {len(TOOLS)} total')
for t in TOOLS:
    print(f'  {t["name"]}')

## Step 4 — execute_tool dispatcher

In [ ]:
from src.agent.dispatcher import make_execute_tool

execute_tool = make_execute_tool(conn)
print('execute_tool dispatcher ready.')

## Step 5 — Schema constant

In [6]:
from src.mcp.schema import GRAPH_SCHEMA_HINT
print(f'GRAPH_SCHEMA_HINT loaded: {len(GRAPH_SCHEMA_HINT)} chars')

GRAPH_SCHEMA_HINT loaded: 7045 chars


## Setup complete

Exported: `conn`, `claude_client`, `oai_client`, `MODEL`, `TOOLS`, `execute_tool`, `GRAPH_SCHEMA_HINT`

Next: Run `notebooks/312_graph_tools.ipynb` to test all tools.